# 01 — 用 create_agent() 一行创建 Agent

> **目标**：理解 Agent 的最简形态 —— 模型 + 工具 + 系统提示词  
> **产出**：一个能调用工具的 Agent，可以问时间和做计算

## 0. 环境准备

确保已在项目根目录执行过：
```bash
pip install -r requirements.txt
```

并在 `.env` 文件中配置了 `OPENAI_API_KEY`。

In [ ]:
# 将项目根目录加入 Python 路径（让 notebook 能 import src 包）
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
print(f"项目根目录：{project_root}")

## 1. 加载配置

In [ ]:
from src.config.settings import settings

print(f"模型：{settings.MODEL_NAME}")
print(f"API Key 前8位：{settings.OPENAI_API_KEY[:8]}...")
print(f"自定义 API 地址：{settings.OPENAI_BASE_URL or '（无，使用官方地址）'}")

## 2. 查看工具定义

工具是 Agent 的"手脚"——没有工具的 Agent 就是一个普通的聊天机器人。
LangChain 的 `@tool` 装饰器会自动把函数的 docstring 和类型标注转换成 LLM 能理解的 function calling schema。

In [ ]:
from src.tools.builtin import ALL_TOOLS

for t in ALL_TOOLS:
    print(f"工具名：{t.name}")
    print(f"描述：{t.description}")
    print(f"参数 schema：{t.args_schema.schema() if t.args_schema else '无参数'}")
    print("-" * 40)

## 3. 创建 Agent

`create_agent()` 是 LangChain v1.0 的核心 API。
它底层建一个 LangGraph StateGraph，内部实现 ReAct 循环（Reason → Act → Observe）。

**参数说明**：
- `model`：ChatModel 实例，负责推理和决策
- `tools`：工具列表，LLM 自主决定何时调用哪个
- `system_prompt`：系统提示词，定义角色和行为边界

In [ ]:
from src.agent.graph import agent

# 查看 agent 的类型 —— 本质是一个编译好的 LangGraph StateGraph
print(type(agent))

## 4. 同步调用 — 问时间

`.invoke()` 是同步调用，等 Agent 全部执行完才返回。

**传入格式**：`{"messages": [{"role": "user", "content": "..."}]}`  
**返回格式**：`{"messages": [...]}`，包含完整对话记录

当用户问"现在几点"时，Agent 内部会：
1. LLM 推理："用户想知道当前时间"
2. LLM 决定："我需要调用 get_current_time 工具"
3. 框架执行 get_current_time()，得到时间字符串
4. 框架把工具结果返回给 LLM
5. LLM 基于工具结果生成自然语言回答

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "现在几点？"}]}
)

# 打印所有消息，看看 Agent 内部的思考过程
for i, msg in enumerate(result["messages"]):
    role = getattr(msg, 'type', 'unknown')
    print(f"[{i}] {role}: {msg.content[:200]}..." if len(str(msg.content)) > 200 else f"[{i}] {role}: {msg.content}")
    # 如果是 AIMessage，检查是否有工具调用
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"     → 调用工具：{tc['name']}({tc['args']})")
    print()

## 5. 同步调用 — 做计算

Agent 能根据用户意图自动选择工具。问计算题时会自动调用 calculator。

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "帮我算一下 (123 + 456) * 7 - 890 等于多少"}]}
)

for i, msg in enumerate(result["messages"]):
    role = getattr(msg, 'type', 'unknown')
    content = str(msg.content)
    if len(content) > 200:
        content = content[:200] + "..."
    print(f"[{i}] {role}: {content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"     → 调用工具：{tc['name']}({tc['args']})")
    print()

## 6. 问无需工具的问题

Agent 不是每次都要调工具 —— 如果 LLM 判断不需要，它直接回答。

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，请介绍一下你自己"}]}
)

for i, msg in enumerate(result["messages"]):
    role = getattr(msg, 'type', 'unknown')
    content = str(msg.content)
    if len(content) > 200:
        content = content[:200] + "..."
    print(f"[{i}] {role}: {content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"     → 调用工具：{tc['name']}({tc['args']})")
    print()

## 7. 流式调用（astream）

`.astream()` 是异步流式调用，适合 Web 应用（可以一边生成一边推送给前端）。
这里用 `async for` 消费事件流。

In [ ]:
async def stream_demo():
    print("=== 开始流式输出 ===\n")
    async for event in agent.astream(
        {"messages": [{"role": "user", "content": "现在几点？请只告诉我时间"}]},
        stream_mode="updates",  # "updates" 只返回状态变更
    ):
        # event 是 {节点名: 状态更新} 的字典
        for node_name, update in event.items():
            if "messages" in update:
                last_msg = update["messages"][-1]
                role = getattr(last_msg, 'type', 'unknown')
                if role == "ai":
                    print(f"[{node_name}] AI: {last_msg.content}")
                elif role == "tool":
                    print(f"[{node_name}] 工具结果: {last_msg.content}")
    print("\n=== 流式输出结束 ===")

import asyncio
await stream_demo()

## 本节小结

| 概念 | 说明 |
|------|------|
| `create_agent()` | LangChain v1.0 的一行创建 Agent API |
| `@tool` | 把普通函数变成 Agent 可调用的工具 |
| `agent.invoke()` | 同步调用，返回完整结果 |
| `agent.astream()` | 异步流式调用，适合 SSE |
| 消息角色 | human(用户) / ai(LLM) / tool(工具返回) |

**下一步**：在 `02_react_loop.ipynb` 中手写 StateGraph，理解 create_agent() 底层到底做了什么。